# V7_B_N02 — Beds, Staff, Supplies: Forecasting Service Capacity

**SRAI Book 7 — Applied AI Studio: National Sector Intelligence**  
Controlled draft. All data are synthetic and illustrative. Analytical outputs require review by the named authority.

## Decision contract
**Decision:** adjust rosters, beds, supplies, referral arrangements, and surge readiness. **Owner:** health-service authority. **Horizon:** daily to monthly. **Boundary:** predictions do not deny care or automate patient-level prioritization.

In [1]:
import numpy as np,pandas as pd
rng=np.random.default_rng(7202); dates=pd.date_range('2026-01-01',periods=180); dow=dates.dayofweek; trend=np.arange(180)*.03; arrivals=np.maximum(0,np.round(42+8*(dow==0)+4*np.sin(2*np.pi*np.arange(180)/7)+trend+rng.normal(0,5,180))).astype(int)
df=pd.DataFrame({'date':dates,'dow':dow,'arrivals':arrivals,'beds_available':rng.integers(38,51,180),'staff_hours':rng.normal(310,25,180).round(1)})
df.tail()

## Evidence and quality contract
An encounter is not necessarily a unique patient. Reporting delay, referral patterns, coding changes, and disruptions affect demand signals. Capacity variables must share the same service boundary and reference period.

In [2]:
assert df.date.is_unique and df.arrivals.ge(0).all(); df['delay_days']=rng.choice([0,1,2,5],len(df),p=[.7,.2,.08,.02]); df['quality_pass']=df.delay_days<=2
print(df.quality_pass.mean(),df[['arrivals','beds_available','staff_hours']].describe().round(1))

0.9833333333333333        arrivals  beds_available  staff_hours
count     180.0           180.0        180.0
mean       46.5            44.7        312.6
std         6.2             3.7         23.1
min        31.0            38.0        250.8
25%        42.0            42.0        298.2
50%        46.5            45.0        314.0
75%        51.0            47.0        329.0
max        64.0            50.0        371.3


## Time-aware demand baselines
Compare recent mean and weekday mean. Random train/test splitting would leak later operational patterns.

In [3]:
train=df.iloc[:150].copy(); test=df.iloc[150:].copy(); test['recent_mean']=train.arrivals.tail(28).mean(); weekday=train.groupby('dow').arrivals.mean(); test['weekday']=test.dow.map(weekday)
def mae(a,p):return float(np.mean(np.abs(np.asarray(a)-np.asarray(p))))
print({'recent_mean_MAE':round(mae(test.arrivals,test.recent_mean),2),'weekday_MAE':round(mae(test.arrivals,test.weekday),2)})

{'recent_mean_MAE': 4.97, 'weekday_MAE': 4.49}


## Trend-plus-weekday candidate and interval
A transparent regression is sufficient for this demonstration. Complexity should increase only if it improves prospective performance and operational value.

In [4]:
X=lambda d:np.column_stack([np.ones(len(d)),np.arange(len(d)),*[np.asarray(d.dow==k,dtype=float) for k in range(1,7)]])
beta=np.linalg.lstsq(X(train),train.arrivals,rcond=None)[0]; Xt=np.column_stack([np.ones(len(test)),np.arange(len(train),len(df)),*[np.asarray(test.dow==k,dtype=float) for k in range(1,7)]]); test['candidate']=Xt@beta; resid=train.arrivals-X(train)@beta; q=np.quantile(np.abs(resid),.90); test['lower']=(test.candidate-q).clip(lower=0); test['upper']=test.candidate+q
print('CANDIDATE_MAE',round(mae(test.arrivals,test.candidate),2),'COVERAGE',round(((test.arrivals>=test.lower)&(test.arrivals<=test.upper)).mean(),2))

CANDIDATE_MAE 3.95 COVERAGE 0.87


## Translate demand into capacity scenarios
Assumptions about service duration and occupancy convert arrivals into workload. These are planning scenarios, not clinical rules.

In [5]:
test['bed_need_reference']=test.candidate*.82; test['bed_need_surge']=test.upper*.90; test['reference_gap']=(test.bed_need_reference-test.beds_available).clip(lower=0); test['surge_gap']=(test.bed_need_surge-test.beds_available).clip(lower=0)
print(test[['date','candidate','lower','upper','beds_available','reference_gap','surge_gap']].tail(7).round(1).to_string(index=False))

      date  candidate  lower  upper  beds_available  reference_gap  surge_gap
2026-06-23       43.7   35.5   51.9              47            0.0        0.0
2026-06-24       44.7   36.5   52.9              44            0.0        3.6
2026-06-25       47.7   39.5   55.8              46            0.0        4.2
2026-06-26       52.0   43.8   60.2              45            0.0        9.2
2026-06-27       51.5   43.3   59.6              46            0.0        7.7
2026-06-28       50.4   42.2   58.5              50            0.0        2.7
2026-06-29       54.2   46.0   62.3              47            0.0        9.1


## Queue and actionability gate
The service plan respects capacity and flags unmet demand. Patient care remains governed by clinical protocols and authorized professionals.

In [6]:
summary={'days_reference_gap':int((test.reference_gap>0).sum()),'days_surge_gap':int((test.surge_gap>0).sum()),'max_surge_gap':round(float(test.surge_gap.max()),1),'actions':['review roster','verify bed state','check supplies','confirm referral capacity'],'prohibited_use':'automated denial or patient-level treatment decision'}
print(summary)

{'days_reference_gap': 7, 'days_surge_gap': 24, 'max_surge_gap': 17.4, 'actions': ['review roster', 'verify bed state', 'check supplies', 'confirm referral capacity'], 'prohibited_use': 'automated denial or patient-level treatment decision'}


## Monitoring and incident trigger
Monitor forecast error, interval coverage, reporting delay, overrides, and unserved demand. A quality breach suspends automated alert generation.

In [7]:
weekly=pd.DataFrame({'week':[1,2,3,4],'MAE':[4.8,5.2,6.1,9.7],'coverage':[.90,.87,.82,.61],'late_rate':[.03,.04,.07,.22]}); weekly['breach']=(weekly.MAE>8)|(weekly.coverage<.75)|(weekly.late_rate>.15)
print(weekly.to_string(index=False))

 week  MAE  coverage  late_rate  breach
    1  4.8      0.90       0.03   False
    2  5.2      0.87       0.04   False
    3  6.1      0.82       0.07   False
    4  9.7      0.61       0.22    True


## Exercises
1. Use rolling-origin evaluation. 2. Add separate emergency and elective demand. 3. Model staff-hours as a constraint. 4. Explain why maximizing occupancy can be unsafe.

## Exact solutions
1. Reforecast from successive dates using only available history and score each horizon. 2. Use distinct concepts, arrival processes, and intervention rules; do not aggregate incompatible demand. 3. Convert workload through documented skill-mix and hours assumptions, then optimize under labour rules. 4. High occupancy removes surge buffer, increases delay and infection/quality risks, and ignores case mix and staffing.

In [8]:
assert test[['candidate','lower','upper','surge_gap']].notna().all().all()
assert weekly.breach.any() and 'denial' in summary['prohibited_use']
print('V7_B_N02_COMPLETE_EXECUTION_PASS')

V7_B_N02_COMPLETE_EXECUTION_PASS
